In [1]:
import regex as re
from collections import defaultdict

In [2]:
corpus  = """low low low low low
lower lower widest widest widest
newest newest newest newest newest newest"""

special_tokens = ["<|endoftext|>"]

In [3]:
vocab = {i: bytes([i]) for i in range(256)} # 0..255
vocab_idx = 256

In [4]:
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
# pretokenize
def pre_tokenize(corpus: str) -> list[str]:
    return re.findall(PAT, corpus)

In [5]:
# count
def get_pre_token_freq(pre_tokens):
    d = defaultdict(int)

    for t in pre_tokens:
        d[tuple(t.encode('utf-8'))] += 1

    return d


In [6]:
def get_byte_pair_freq(d):
    byte_pair_freq = defaultdict(int)

    for tok_bytes, freq in d.items():
        for i in range(len(tok_bytes)-1):
            byte_pair_freq[(tok_bytes[i], tok_bytes[i+1])] += freq

    return byte_pair_freq

In [7]:
def merge(d, best_pair, vocab_idx):
    new_d = defaultdict(int)

    for tok_bytes, freq in d.items():
        new_bytes = []
        i = 0
        while i < len(tok_bytes):
            if i != len(tok_bytes)-1 and (tok_bytes[i], tok_bytes[i+1]) == best_pair:
                new_bytes.append(vocab_idx)
                i+=2
            else:
                new_bytes.append(tok_bytes[i])
                i+=1

        new_d[tuple(new_bytes)] += freq
    return new_d

In [8]:
pre_tokens = pre_tokenize(corpus)

In [9]:

merges = []
merge_cnt = 60

d = get_pre_token_freq(pre_tokens)
byte_pair_freq = get_byte_pair_freq(d)
best_pair = max(byte_pair_freq, key=byte_pair_freq.get)


In [10]:
best_pair

(101, 115)

In [11]:
for i in range(merge_cnt):
    print(f"{i}th merge:", best_pair, vocab[best_pair[0]], vocab[best_pair[1]])

    merges.append(best_pair)
    vocab[vocab_idx] = vocab[best_pair[0]] + vocab[best_pair[1]]

    d = merge(d, best_pair, vocab_idx)
    vocab_idx += 1

    byte_pair_freq = get_byte_pair_freq(d)
    if not byte_pair_freq:
        break
    best_pair = max(byte_pair_freq, key=byte_pair_freq.get)

0th merge: (101, 115) b'e' b's'
1th merge: (256, 116) b'es' b't'
2th merge: (108, 111) b'l' b'o'
3th merge: (258, 119) b'lo' b'w'
4th merge: (110, 101) b'n' b'e'
5th merge: (260, 119) b'ne' b'w'
6th merge: (261, 257) b'new' b'est'
7th merge: (32, 259) b' ' b'low'
8th merge: (32, 262) b' ' b'newest'
9th merge: (32, 119) b' ' b'w'
10th merge: (265, 105) b' w' b'i'
11th merge: (266, 100) b' wi' b'd'
12th merge: (267, 257) b' wid' b'est'
13th merge: (101, 114) b'e' b'r'
14th merge: (259, 269) b'low' b'er'
15th merge: (263, 269) b' low' b'er'


In [12]:
merges

[(101, 115),
 (256, 116),
 (108, 111),
 (258, 119),
 (110, 101),
 (260, 119),
 (261, 257),
 (32, 259),
 (32, 262),
 (32, 119),
 (265, 105),
 (266, 100),
 (267, 257),
 (101, 114),
 (259, 269),
 (263, 269)]

In [13]:
for idx in range(251, 256):
    print(idx, vocab[idx])
print()
for idx in range(256, vocab_idx):
    print(idx, vocab[idx])

251 b'\xfb'
252 b'\xfc'
253 b'\xfd'
254 b'\xfe'
255 b'\xff'

256 b'es'
257 b'est'
258 b'lo'
259 b'low'
260 b'ne'
261 b'new'
262 b'newest'
263 b' low'
264 b' newest'
265 b' w'
266 b' wi'
267 b' wid'
268 b' widest'
269 b'er'
270 b'lower'
271 b' lower'
